# Vegetation, Built-up, and Bare Soil Indices — Ouagadougou (2024)

**Data:** Sentinel-2 Surface Reflectance (`COPERNICUS/S2_SR_HARMONIZED`)  
**Study period:** 1 March – 30 May 2024 (heatwave season)  
**Native resolution:** 10 m · **Export resolution:** 30 m  
**Cloud masking:** QA60 band (clouds + cirrus); images with < 30 % valid pixels discarded

---

## Overview

This notebook computes three spectral indices over Ouagadougou using a cloud-free Sentinel-2 median composite, for use as surface-cover predictors in heatwave hotspot analysis.

| Index | Formula | Interpretation |
|---|---|---|
| **NDVI** | (NIR − Red) / (NIR + Red) | Vegetation density; negative in built-up/bare areas |
| **NDBI** | (SWIR − NIR) / (SWIR + NIR) | Built-up intensity; positive in urban surfaces |
| **BSI** | ((SWIR + Red) − (NIR + Blue)) / ((SWIR + Red) + (NIR + Blue)) | Bare soil exposure |

The workflow has six steps:

1. **Setup** — configuration and Earth Engine initialisation  
2. **AOI** — load Ouagadougou shapefile and convert to EE geometry  
3. **Image collection** — filter, cloud-mask, and quality-check Sentinel-2 scenes  
4. **Composite & indices** — median composite, NDVI / NDBI / BSI calculation  
5. **Export** — resample to 30 m and write GeoTIFF files  
6. **Validation** — per-file statistics and missing-data assessment

> **Output directory:** `../../data/raw/ndvi_ndbi_bsi/`

---
## 1 · Setup

In [ ]:
import ee
import geemap
import geopandas as gpd
import rasterio
import numpy as np
import time
import sys
from pathlib import Path
from datetime import datetime

print('Libraries imported.')

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
CONFIG = {
    'ee_project':        '183927144814',
    'shapefile_path':    '../../data/raw/Shapefile Ouaga/Ouaga.shp',
    'output_dir':        Path('../../data/raw/ndvi_ndbi_bsi'),
    'start_date':        '2024-03-01',
    'end_date':          '2024-05-30',
    'native_scale':      10,    # Sentinel-2 native resolution (m)
    'export_scale':      30,    # Export resolution (m)
    'valid_px_threshold': 0.30, # Minimum valid-pixel fraction to keep an image
    'map_center':        [12.37, -1.53],
    'map_zoom':          11,
}
CONFIG['output_dir'].mkdir(parents=True, exist_ok=True)

# ── Initialise Earth Engine ───────────────────────────────────────────────────
try:
    ee.Initialize(project=CONFIG['ee_project'])
    print('Earth Engine initialised successfully.')
except Exception as e:
    print(f'Error initialising Earth Engine: {e}')
    sys.exit(1)

---
## 2 · Area of Interest

The Ouagadougou administrative boundary is loaded from a local shapefile, dissolved to a single polygon, and converted to an Earth Engine geometry.

In [ ]:
try:
    aoi_gdf  = gpd.read_file(CONFIG['shapefile_path']).to_crs(epsg=4326)
    aoi_gdf  = aoi_gdf.dissolve()
    coords   = aoi_gdf.geometry.iloc[0].__geo_interface__['coordinates']
    aoi_geom = ee.Geometry.Polygon(coords)
    print('AOI loaded and converted to EE geometry.')
    print(f'Bounding box: {aoi_gdf.total_bounds.round(4)}')
except FileNotFoundError:
    print(f'Shapefile not found: {CONFIG["shapefile_path"]}')
    sys.exit(1)

---
## 3 · Image Collection — Filtering and Quality Control

### 3.1 · Load and Cloud-Mask

In [ ]:
def mask_s2_clouds(image):
    """
    Mask clouds and cirrus using the QA60 band and scale reflectance to [0, 1].
    Key metadata (system:time_start, system:index) is preserved.
    """
    qa         = image.select('QA60')
    cloud_bit  = 1 << 10
    cirrus_bit = 1 << 11
    mask = (
        qa.bitwiseAnd(cloud_bit).eq(0)
          .And(qa.bitwiseAnd(cirrus_bit).eq(0))
    )
    return (
        image.divide(10000)
             .updateMask(mask)
             .copyProperties(image, ['system:time_start', 'system:index'])
    )


# Load and cloud-mask the collection
s2_raw    = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
               .filterBounds(aoi_geom)
               .filterDate(CONFIG['start_date'], CONFIG['end_date']))
s2_masked = s2_raw.map(mask_s2_clouds)

n_raw = s2_raw.size().getInfo()
print(f'Sentinel-2 scenes found in AOI / period: {n_raw}')
if n_raw == 0:
    print('No images found. Check date range and AOI.')
    sys.exit(1)

### 3.2 · Per-Image Quality Assessment

Each scene is assessed for valid-pixel coverage across all bands used in index calculation (B2, B4, B8, B11). Images where any band falls below the 30 % threshold are discarded.

In [ ]:
INDEX_BANDS = ['B2', 'B4', 'B8', 'B11']
THRESHOLD   = CONFIG['valid_px_threshold']

image_list = s2_masked.toList(s2_masked.size())
n_images   = s2_masked.size().getInfo()

print(f'Per-band quality assessment — {n_images} images')
print(f'Threshold: all bands must have >= {THRESHOLD*100:.0f}% valid pixels')
print('=' * 115)
print(f'{"Img":>4} | {"Date":^10} | {"B2":>6} | {"B4":>6} | {"B8":>6} | {"B11":>6} | {"Min":>6} | {"Decision":^8} | {"Bottleneck":^12}')
print('-' * 115)

kept_indices   = []
kept_valid     = []
skipped_images = []
start_time     = time.time()

# Total pixel count is the same for all bands and images
total_px = (
    ee.Image.constant(1).clip(aoi_geom)
      .reduceRegion(reducer=ee.Reducer.count(), geometry=aoi_geom,
                    scale=10, maxPixels=1e13, bestEffort=True)
      .get('constant')
)
total_px = ee.Number(total_px).getInfo()

for i in range(n_images):
    try:
        img  = ee.Image(image_list.get(i))
        ts   = img.get('system:time_start').getInfo()
        date = datetime.fromtimestamp(ts / 1000).strftime('%Y-%m-%d') if ts else 'N/A'

        band_fracs = {}
        for band in INDEX_BANDS:
            valid_px = (
                img.select(band).mask()
                   .reduceRegion(reducer=ee.Reducer.sum(), geometry=aoi_geom,
                                 scale=10, maxPixels=1e13, bestEffort=True)
                   .get(band)
            )
            valid_px       = ee.Number(valid_px).getInfo() if valid_px else 0
            band_fracs[band] = valid_px / total_px if total_px > 0 else 0

        min_frac   = min(band_fracs.values())
        worst_band = min(band_fracs, key=band_fracs.get)
        status     = 'KEEP' if min_frac >= THRESHOLD else 'SKIP'
        bottleneck = f"{worst_band}={band_fracs[worst_band]*100:.1f}%" if status == 'SKIP' else 'All OK'

        print(
            f'{i+1:4d} | {date:^10} | '
            f"{band_fracs['B2']*100:5.1f}% | {band_fracs['B4']*100:5.1f}% | "
            f"{band_fracs['B8']*100:5.1f}% | {band_fracs['B11']*100:5.1f}% | "
            f'{min_frac*100:5.1f}% | {status:^8} | {bottleneck:^12}'
        )

        if status == 'KEEP':
            kept_indices.append(i)
            kept_valid.append(min_frac)
        else:
            skipped_images.append((date, worst_band, band_fracs[worst_band]))

        if (i + 1) % 5 == 0:
            time.sleep(1)   # avoid GEE rate limits

    except Exception as e:
        print(f'{i+1:4d} | ERROR: {e}')

elapsed = time.time() - start_time
print('-' * 115)
print(f'Processing time : {elapsed:.1f} s')
print(f'Images retained : {len(kept_indices)}/{n_images}')
print(f'Images discarded: {len(skipped_images)}/{n_images}')

if skipped_images:
    band_problems = {}
    for _, band, frac in skipped_images:
        band_problems.setdefault(band, []).append(frac)
    print('\nBottleneck bands:')
    for band, fracs in sorted(band_problems.items()):
        print(f'  {band}: bottleneck in {len(fracs)} image(s), avg coverage {np.mean(fracs)*100:.1f}%')

# Build filtered collection
s2_clean = ee.ImageCollection([ee.Image(image_list.get(i)) for i in kept_indices])
print(f'\nFiltered collection ready: {len(kept_indices)} scenes.')

---
## 4 · Median Composite and Index Calculation

A pixel-wise median is computed across all retained scenes. The median is preferred over the mean because it is robust to residual cloud contamination and outliers.

In [ ]:
# ── Median composite ──────────────────────────────────────────────────────────
median_img = s2_clean.median().clip(aoi_geom)
print(f'Median composite created from {len(kept_indices)} scenes.')

# ── Spectral indices ──────────────────────────────────────────────────────────
# NDVI: (NIR - Red) / (NIR + Red)
ndvi = median_img.normalizedDifference(['B8', 'B4']).rename('NDVI')

# NDBI: (SWIR - NIR) / (SWIR + NIR)
ndbi = median_img.normalizedDifference(['B11', 'B8']).rename('NDBI')

# BSI: ((SWIR + Red) - (NIR + Blue)) / ((SWIR + Red) + (NIR + Blue))
bsi = median_img.expression(
    '((swir + red) - (nir + blue)) / ((swir + red) + (nir + blue))',
    {
        'swir': median_img.select('B11'),
        'red':  median_img.select('B4'),
        'nir':  median_img.select('B8'),
        'blue': median_img.select('B2'),
    }
).rename('BSI')

print('NDVI, NDBI, and BSI computed.')

# ── Value range validation ────────────────────────────────────────────────────
def get_stats(img, band):
    return img.select(band).reduceRegion(
        reducer=ee.Reducer.minMax(),
        geometry=aoi_geom, scale=10, maxPixels=1e13
    ).getInfo()

print('\nIndex value ranges (native 10 m):')
for name, img in [('NDVI', ndvi), ('NDBI', ndbi), ('BSI', bsi)]:
    s = get_stats(img, name)
    print(f'  {name}: [{s[name+"_min"]:.3f}, {s[name+"_max"]:.3f}]')

### 4.1 · Interactive Map Preview

In [ ]:
# ── Resample to 30 m for display and export ───────────────────────────────────
crs     = median_img.select('B4').projection().crs()
ndvi_30 = ndvi.reproject(crs=crs, scale=CONFIG['export_scale'])
ndbi_30 = ndbi.reproject(crs=crs, scale=CONFIG['export_scale'])
bsi_30  = bsi.reproject(crs=crs, scale=CONFIG['export_scale'])
print('Indices resampled to 30 m.')

# ── Map ───────────────────────────────────────────────────────────────────────
m = geemap.Map(center=CONFIG['map_center'], zoom=CONFIG['map_zoom'])

ndvi_vis = {'min': -1,   'max': 1,   'palette': ['red', 'yellow', 'green']}
ndbi_vis = {'min': -0.5, 'max': 0.5, 'palette': ['green', 'white', 'gray']}
bsi_vis  = {'min': -0.5, 'max': 0.5, 'palette': ['blue', 'white', 'brown']}

m.addLayer(ndvi_30, ndvi_vis, 'NDVI (30 m)')
m.addLayer(ndbi_30, ndbi_vis, 'NDBI (30 m)')
m.addLayer(bsi_30,  bsi_vis,  'BSI (30 m)')
m.addLayer(ndvi,    ndvi_vis, 'NDVI (10 m)', shown=False)
m.addLayer(ndbi,    ndbi_vis, 'NDBI (10 m)', shown=False)
m.addLayer(bsi,     bsi_vis,  'BSI  (10 m)', shown=False)
m.addLayer(aoi_geom, {'color': 'red'}, 'AOI boundary')

print('Map ready — toggle layers in the control panel to compare resolutions.')
m

---
## 5 · Export to GeoTIFF

In [ ]:
indices_to_export = {'NDVI': ndvi_30, 'NDBI': ndbi_30, 'BSI': bsi_30}

print(f'Exporting at {CONFIG["export_scale"]} m resolution...')
for name, img in indices_to_export.items():
    out_path = CONFIG['output_dir'] / f'{name}_{CONFIG["export_scale"]}m.tif'
    try:
        geemap.ee_export_image(
            img.clip(aoi_geom),
            filename=str(out_path),
            scale=CONFIG['export_scale'],
            region=aoi_geom,
            file_per_band=False,
        )
        print(f'  {name} exported -> {out_path}')
    except Exception as e:
        print(f'  {name} export failed: {e}')
        print('    Alternative: use ee.batch.Export.image.toDrive()')

---
## 6 · Validation

### 6.1 · Per-File Statistics and Missing-Data Assessment

In [ ]:
print('=' * 65)
print('EXPORTED FILE VALIDATION')
print('=' * 65)

for name in ['NDVI', 'NDBI', 'BSI']:
    fpath = CONFIG['output_dir'] / f'{name}_{CONFIG["export_scale"]}m.tif'

    if not fpath.exists():
        print(f'\n{name}: file not found at {fpath}')
        continue

    with rasterio.open(fpath) as src:
        data    = src.read(1)
        nodata  = src.nodata
        total   = data.size
        n_nan   = int(np.sum(np.isnan(data)))
        n_nd    = int(np.sum(data == nodata)) if nodata is not None else 0
        missing = n_nan + n_nd
        valid   = total - missing
        miss_pct = 100 * missing / total

        quality = (
            'Excellent' if miss_pct < 5  else
            'Good'      if miss_pct < 15 else
            'Fair'      if miss_pct < 30 else
            'Poor'
        )

        print(f'\n{name}_30m.tif')
        print(f'  Shape       : {data.shape}')
        print(f'  CRS         : {src.crs}')
        print(f'  Value range : [{np.nanmin(data):.3f}, {np.nanmax(data):.3f}]')
        print(f'  Valid pixels: {valid:,} / {total:,} ({100*valid/total:.1f}%)')
        print(f'  Missing     : {missing:,} ({miss_pct:.1f}%)')
        print(f'  Quality     : {quality}')

---
## Summary

| Output file | Description |
|---|---|
| `NDVI_30m.tif` | Normalised Difference Vegetation Index |
| `NDBI_30m.tif` | Normalised Difference Built-up Index |
| `BSI_30m.tif`  | Bare Soil Index |

All files are in the native Sentinel-2 UTM projection, resampled to 30 m by mean aggregation. They are ready for overlay with ERA5-Land temperature fields or as predictors in a hotspot regression model.

---

### Methodology Note

Sentinel-2 Level-2A (surface reflectance) imagery is retrieved from `COPERNICUS/S2_SR_HARMONIZED` for the heatwave period (March–May 2024). Clouds and cirrus are masked using the QA60 bitmask. Scenes where any index band (B2, B4, B8, B11) has less than 30 % valid pixels within the AOI are discarded. A pixel-wise temporal median is then computed across the retained scenes to produce a single cloud-free composite.

